# Topological Analysis of LLM Representation Sparsity Under OOD Shift

**Premise**: Jin et al. (arXiv:2603.03415) show that harder tasks produce sparser last-hidden-state
representations in LLMs. We apply persistent homology (via ATT) to ask whether the *geometry*
of those representations also changes — and whether topology captures structure that scalar
sparsity metrics miss.

**Four experiments**:
1. Point cloud topology of hidden state ensembles per difficulty level
2. Token-position trajectory topology (point cloud approach, no Takens)
3. Layer-wise topological transition profile (Level 1 vs Level 5)
4. Pairwise topological distance between all difficulty levels

**Prerequisites**: Run `scripts/extract_hidden_states.py` first to produce the `.npz` file.

In [ ]:
import os
import numpy as np
from scipy import stats
from sklearn.decomposition import PCA
from att.config import set_seed
from att.topology import PersistenceAnalyzer
from att.embedding import validate_embedding
from att.viz import plot_persistence_diagram, plot_persistence_image, plot_binding_image
import matplotlib.pyplot as plt

set_seed(42)

FIGURES_DIR = os.path.join(os.path.dirname(os.getcwd()), "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

# --- Load extracted hidden states ---
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), "data", "transformer", "math500_hidden_states.npz")
data = np.load(DATA_PATH, allow_pickle=True)

last_hidden = data["last_hidden_states"]       # (N, d)
levels = data["difficulty_levels"]              # (N,)
layer_hidden = data["layer_hidden_states"]      # (N, L+1, d)
token_trajs = data["token_trajectories"]        # object array of (T_i, d) arrays
seq_lengths = data["seq_lengths"]               # (N,)
model_name = str(data["model_name"])
hidden_dim = int(data["hidden_dim"])
num_layers = int(data["num_layers"])

print(f"Model: {model_name}")
print(f"Hidden dim: {hidden_dim}, Layers (incl. embedding): {num_layers}")
print(f"Total problems: {len(last_hidden)}")
for lv in range(1, 6):
    print(f"  Level {lv}: {(levels == lv).sum()} problems")

## Cell 0: Sparsity Baselines (Jin et al. replication)

Compute L1 norm, Top-10% Energy, and Effective Rank per difficulty level.
If the sparsity-difficulty trend does not appear, the model/data is unsuitable and
topology experiments will not be meaningful.

In [ ]:
def l1_norm(h):
    """Mean absolute activation (per-dimension L1 norm)."""
    return np.sum(np.abs(h)) / len(h)


def top_k_energy(h, k_pct=0.1):
    """Fraction of total squared energy in top k% of dimensions."""
    h2 = h ** 2
    total = np.sum(h2)
    if total < 1e-15:
        return 0.0
    k = max(1, int(len(h) * k_pct))
    top_k = np.sort(h2)[-k:]
    return np.sum(top_k) / total


def effective_rank(h):
    """Normalized exponential of spectral entropy (0 to 1 scale)."""
    h2 = h ** 2
    total = np.sum(h2)
    if total < 1e-15:
        return 0.0
    p = h2 / total
    p = p[p > 0]
    entropy = -np.sum(p * np.log(p))
    return np.exp(entropy) / len(h)


# Compute per-level aggregates
sparsity_metrics = {}
per_problem_metrics = {"l1_norm": [], "top10_energy": [], "eff_rank": [], "level": []}

for level in range(1, 6):
    mask = levels == level
    hs = last_hidden[mask]
    l1s = [l1_norm(h) for h in hs]
    t10s = [top_k_energy(h, 0.1) for h in hs]
    ers = [effective_rank(h) for h in hs]

    sparsity_metrics[level] = {
        "l1_norm": np.mean(l1s),
        "top10_energy": np.mean(t10s),
        "eff_rank": np.mean(ers),
        "n_problems": int(mask.sum()),
    }

    per_problem_metrics["l1_norm"].extend(l1s)
    per_problem_metrics["top10_energy"].extend(t10s)
    per_problem_metrics["eff_rank"].extend(ers)
    per_problem_metrics["level"].extend([level] * len(l1s))

    print(
        f"Level {level}: n={mask.sum()}, "
        f"L1={sparsity_metrics[level]['l1_norm']:.4f}, "
        f"Top10E={sparsity_metrics[level]['top10_energy']:.4f}, "
        f"EffRank={sparsity_metrics[level]['eff_rank']:.4f}"
    )

# Quick sanity: does Top10% Energy increase with difficulty? (Jin et al. prediction)
t10_values = [sparsity_metrics[lv]["top10_energy"] for lv in range(1, 6)]
r_t10, p_t10 = stats.spearmanr(range(1, 6), t10_values)
print(f"\nSpearman(difficulty, Top10E): r={r_t10:.3f}, p={p_t10:.4f}")
if r_t10 > 0 and p_t10 < 0.1:
    print("Sparsity-difficulty trend CONFIRMED. Proceeding with topology experiments.")
else:
    print("WARNING: Sparsity-difficulty trend NOT confirmed. Results may be uninterpretable.")

## Experiment 1: Point Cloud Topology of Hidden State Ensembles

For each difficulty level, treat the set of last hidden states as a point cloud in R^d.
PCA to 50 components, then run persistent homology (H0, H1, H2).
Ask: does the topology of the representation ensemble change with difficulty?

In [ ]:
# --- Experiment 1: PCA + Persistent Homology per difficulty level ---

N_PCA_COMPONENTS = 50

exp1_results = {}   # level -> PersistenceAnalyzer result dict
exp1_analyzers = {} # level -> PersistenceAnalyzer (for distance computation later)
exp1_pca_info = {}  # level -> cumulative variance ratios

for level in range(1, 6):
    mask = levels == level
    cloud = last_hidden[mask]  # (n_k, d)
    n_k = cloud.shape[0]

    # PCA reduction
    n_components = min(N_PCA_COMPONENTS, n_k - 1, cloud.shape[1])
    pca = PCA(n_components=n_components)
    cloud_pca = pca.fit_transform(cloud)  # (n_k, n_components)

    cumvar = pca.explained_variance_ratio_.cumsum()
    exp1_pca_info[level] = cumvar

    print(
        f"Level {level}: {n_k} points -> PCA to {n_components}d, "
        f"variance explained: {cumvar[-1]:.4f} "
        f"(@10={cumvar[min(9, len(cumvar)-1)]:.4f}, "
        f"@20={cumvar[min(19, len(cumvar)-1)]:.4f})"
    )

    # Persistent homology
    pa = PersistenceAnalyzer(max_dim=2, backend="ripser")
    result = pa.fit_transform(cloud_pca, subsample=min(n_k, 200), seed=42)

    exp1_results[level] = result
    exp1_analyzers[level] = pa

    # Report persistence entropy
    pe = result["persistence_entropy"]
    print(
        f"  Persistence entropy: H0={pe[0]:.4f}, H1={pe[1]:.4f}, "
        f"H2={pe[2]:.4f}"
    )
    print(
        f"  Bottleneck norms: H0={result['bottleneck_norms'][0]:.4f}, "
        f"H1={result['bottleneck_norms'][1]:.4f}, "
        f"H2={result['bottleneck_norms'][2]:.4f}"
    )
    print()

In [ ]:
# --- Experiment 1: Plots ---

# 1a. 5-panel persistence diagrams (H0 + H1)
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for i, level in enumerate(range(1, 6)):
    ax = axes[i]
    diagrams = exp1_results[level]["diagrams"]
    # Plot H0 and H1 only (skip H2 for clarity)
    colors = ["tab:blue", "tab:orange"]
    for dim in range(2):
        dgm = diagrams[dim]
        if len(dgm) > 0:
            ax.scatter(
                dgm[:, 0], dgm[:, 1],
                c=colors[dim], label=f"H{dim}",
                s=15, alpha=0.7, edgecolors="k", linewidths=0.3,
            )
    # Diagonal
    all_vals = np.concatenate([d.ravel() for d in diagrams[:2] if len(d) > 0])
    if len(all_vals) > 0:
        vmin, vmax = all_vals.min(), all_vals.max()
        ax.plot([vmin, vmax], [vmin, vmax], "k--", alpha=0.3)
    ax.set_title(f"Level {level}")
    ax.set_xlabel("Birth")
    if i == 0:
        ax.set_ylabel("Death")
    ax.legend(fontsize=8)
    ax.set_aspect("equal")

fig.suptitle("Persistence Diagrams by Difficulty Level", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "exp1_persistence_diagrams.png"), dpi=300, bbox_inches="tight")
plt.show()

# 1b. Persistence entropy H1 vs difficulty
pe_h1 = [exp1_results[lv]["persistence_entropy"][1] for lv in range(1, 6)]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(range(1, 6), pe_h1, "o-", color="tab:red", linewidth=2, markersize=8)
ax.set_xlabel("Difficulty Level")
ax.set_ylabel("Persistence Entropy (H1)")
ax.set_title("H1 Persistence Entropy vs Task Difficulty")
ax.set_xticks(range(1, 6))
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "exp1_entropy_vs_difficulty.png"), dpi=300, bbox_inches="tight")
plt.show()

# 1c. Cumulative PCA variance at 10, 20, 50 components vs difficulty
fig, ax = plt.subplots(figsize=(6, 4))
for n_comp, color, label in [(10, "tab:blue", "@10"), (20, "tab:orange", "@20"), (50, "tab:green", "@50")]:
    vals = []
    for lv in range(1, 6):
        cv = exp1_pca_info[lv]
        idx = min(n_comp - 1, len(cv) - 1)
        vals.append(cv[idx])
    ax.plot(range(1, 6), vals, "o-", color=color, linewidth=2, markersize=8, label=label)
ax.set_xlabel("Difficulty Level")
ax.set_ylabel("Cumulative Variance Explained")
ax.set_title("PCA Variance Concentration vs Difficulty")
ax.set_xticks(range(1, 6))
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "exp1_pca_variance.png"), dpi=300, bbox_inches="tight")
plt.show()

# 1d. Persistence image difference: PI(Level5) - PI(Level1) for H1
# Compute images on a shared grid for comparability
all_h1_births = []
all_h1_pers = []
for lv in range(1, 6):
    dgm = exp1_results[lv]["diagrams"][1]  # H1
    if len(dgm) > 0:
        all_h1_births.extend(dgm[:, 0])
        all_h1_pers.extend(dgm[:, 1] - dgm[:, 0])

if all_h1_births:
    shared_birth_range = (min(all_h1_births), max(all_h1_births))
    shared_pers_range = (0.0, max(all_h1_pers))
else:
    shared_birth_range = (0.0, 1.0)
    shared_pers_range = (0.0, 1.0)

pi_level1 = exp1_analyzers[1].to_image(
    resolution=50, sigma=0.1,
    birth_range=shared_birth_range, persistence_range=shared_pers_range,
)
pi_level5 = exp1_analyzers[5].to_image(
    resolution=50, sigma=0.1,
    birth_range=shared_birth_range, persistence_range=shared_pers_range,
)

# H1 difference
diff_h1 = pi_level5[1] - pi_level1[1]  # index 1 = H1

fig, ax = plt.subplots(figsize=(6, 5))
vmax = max(abs(diff_h1.min()), abs(diff_h1.max())) or 1.0
im = ax.imshow(diff_h1, cmap="RdBu_r", origin="lower", aspect="auto", vmin=-vmax, vmax=vmax)
ax.set_title("PI(Level 5) - PI(Level 1), H1")
ax.set_xlabel("Birth")
ax.set_ylabel("Persistence")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "exp1_pi_difference_h1.png"), dpi=300, bbox_inches="tight")
plt.show()

# 1e. Correlation: persistence entropy H1 vs sparsity metrics
level_pe_h1 = np.array([exp1_results[lv]["persistence_entropy"][1] for lv in range(1, 6)])
level_l1 = np.array([sparsity_metrics[lv]["l1_norm"] for lv in range(1, 6)])
level_t10 = np.array([sparsity_metrics[lv]["top10_energy"] for lv in range(1, 6)])
level_er = np.array([sparsity_metrics[lv]["eff_rank"] for lv in range(1, 6)])

print("Pearson correlations (5 data points, one per difficulty level):")
for name, vals in [("L1 Norm", level_l1), ("Top10% Energy", level_t10), ("Eff Rank", level_er)]:
    r, p = stats.pearsonr(level_pe_h1, vals)
    print(f"  Persistence Entropy H1 vs {name}: r={r:.3f}, p={p:.4f}")

## Experiment 2: Trajectory Topology (Point Cloud, No Takens)

For each problem, treat the token-position hidden states at the final layer as a point
cloud (each token = one point in PCA-reduced space). Run PH and embedding validation.
Ask: do harder problems produce trajectories with simpler/more degenerate geometry?

In [ ]:
# --- Experiment 2: Token trajectory topology per problem ---

N_PROBLEMS_PER_LEVEL = 20
N_PCA_TRAJ = 10

exp2_results = {lv: [] for lv in range(1, 6)}  # level -> list of per-problem dicts

for level in range(1, 6):
    mask = levels == level
    level_indices = np.where(mask)[0][:N_PROBLEMS_PER_LEVEL]

    n_degen = 0
    n_total = 0

    for idx in level_indices:
        traj = token_trajs[idx]  # (T_i, d)
        if traj.shape[0] < 5:
            # Too few tokens for meaningful topology
            continue

        n_total += 1

        # PCA to 10 components
        n_comp = min(N_PCA_TRAJ, traj.shape[0] - 1, traj.shape[1])
        pca = PCA(n_components=n_comp)
        traj_pca = pca.fit_transform(traj)  # (T_i, n_comp)

        # Embedding validation
        val = validate_embedding(traj_pca)
        is_degen = val["degenerate"]
        if is_degen:
            n_degen += 1

        # Persistent homology
        pa = PersistenceAnalyzer(max_dim=1, backend="ripser")
        result = pa.fit_transform(traj_pca, subsample=min(traj_pca.shape[0], 500), seed=42)

        exp2_results[level].append({
            "persistence_entropy_h1": result["persistence_entropy"][1],
            "condition_number": val["condition_number"],
            "effective_rank": val["effective_rank"],
            "degenerate": is_degen,
            "n_tokens": traj.shape[0],
            "n_h1_features": len(result["diagrams"][1]),
        })

    degen_rate = n_degen / n_total if n_total > 0 else 0
    mean_cond = np.mean([r["condition_number"] for r in exp2_results[level]]) if exp2_results[level] else 0
    mean_pe = np.mean([r["persistence_entropy_h1"] for r in exp2_results[level]]) if exp2_results[level] else 0
    print(
        f"Level {level}: {n_total} trajectories, "
        f"{degen_rate:.0%} degenerate, "
        f"mean cond={mean_cond:.1f}, "
        f"mean PE(H1)={mean_pe:.4f}"
    )

In [ ]:
# --- Experiment 2: Plots ---

level_labels = list(range(1, 6))

# Collect per-level statistics
exp2_pe_means = []
exp2_pe_stds = []
exp2_cond_means = []
exp2_cond_stds = []
exp2_degen_rates = []

for lv in level_labels:
    results = exp2_results[lv]
    if results:
        pes = [r["persistence_entropy_h1"] for r in results]
        conds = [r["condition_number"] for r in results]
        degens = [r["degenerate"] for r in results]
        exp2_pe_means.append(np.mean(pes))
        exp2_pe_stds.append(np.std(pes) / np.sqrt(len(pes)))
        exp2_cond_means.append(np.mean(conds))
        exp2_cond_stds.append(np.std(conds) / np.sqrt(len(conds)))
        exp2_degen_rates.append(np.mean(degens))
    else:
        exp2_pe_means.append(0)
        exp2_pe_stds.append(0)
        exp2_cond_means.append(0)
        exp2_cond_stds.append(0)
        exp2_degen_rates.append(0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 2a. Mean persistence entropy H1 vs difficulty
ax = axes[0]
ax.errorbar(level_labels, exp2_pe_means, yerr=exp2_pe_stds,
            fmt="o-", color="tab:red", linewidth=2, markersize=8, capsize=4)
ax.set_xlabel("Difficulty Level")
ax.set_ylabel("Persistence Entropy (H1)")
ax.set_title("Trajectory H1 Entropy vs Difficulty")
ax.set_xticks(level_labels)
ax.grid(True, alpha=0.3)

# 2b. Mean condition number vs difficulty
ax = axes[1]
ax.errorbar(level_labels, exp2_cond_means, yerr=exp2_cond_stds,
            fmt="s-", color="tab:purple", linewidth=2, markersize=8, capsize=4)
ax.set_xlabel("Difficulty Level")
ax.set_ylabel("Condition Number")
ax.set_title("Trajectory Condition Number vs Difficulty")
ax.set_xticks(level_labels)
ax.grid(True, alpha=0.3)

# 2c. Degeneracy rate bar chart
ax = axes[2]
bars = ax.bar(level_labels, exp2_degen_rates, color="tab:gray", edgecolor="black")
ax.set_xlabel("Difficulty Level")
ax.set_ylabel("Fraction Degenerate")
ax.set_title("Trajectory Degeneracy Rate vs Difficulty")
ax.set_xticks(level_labels)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis="y")
for bar, rate in zip(bars, exp2_degen_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{rate:.0%}", ha="center", va="bottom", fontsize=10)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "exp2_trajectory_topology.png"), dpi=300, bbox_inches="tight")
plt.show()

# Print summary table
print("Level | Degen Rate | Mean Cond | Mean PE(H1) | Mean #H1 Features")
print("------|------------|-----------|-------------|------------------")
for lv in level_labels:
    results = exp2_results[lv]
    if results:
        dr = np.mean([r["degenerate"] for r in results])
        mc = np.mean([r["condition_number"] for r in results])
        mp = np.mean([r["persistence_entropy_h1"] for r in results])
        mf = np.mean([r["n_h1_features"] for r in results])
        print(f"  {lv}   | {dr:>9.0%} | {mc:>9.1f} | {mp:>11.4f} | {mf:>17.1f}")
    else:
        print(f"  {lv}   | no data")

## Experiment 3: Layer-wise Topological Transition Profile

For Level 1 and Level 5 (extremes), collect the hidden state at each layer for 30 problems,
forming a point cloud per layer. Run PH per layer, then compute bottleneck distance between
consecutive layers. Ask: does the topological transition concentrate in the final layers?
Does Level 5 show a sharper terminal transition than Level 1?

In [ ]:
# --- Experiment 3: Layer-wise topological profile ---

N_PROBLEMS_EXP3 = 30
N_PCA_LAYER = 20
LEVELS_EXP3 = [1, 5]

exp3_distances = {}  # level -> list of bottleneck distances between consecutive layers
exp3_entropies = {}  # level -> list of H1 persistence entropy per layer

for level in LEVELS_EXP3:
    mask = levels == level
    level_indices = np.where(mask)[0][:N_PROBLEMS_EXP3]
    n_problems = len(level_indices)

    # layer_hidden shape: (N, num_layers, d)
    # num_layers includes embedding layer (index 0) through final layer (index num_layers-1)
    n_total_layers = layer_hidden.shape[1]

    print(f"Level {level}: {n_problems} problems, {n_total_layers} layers")

    analyzers_per_layer = []
    layer_entropies = []

    for ell in range(n_total_layers):
        # Collect layer-ell hidden state for all selected problems -> (n_problems, d)
        cloud = layer_hidden[level_indices, ell, :]  # (n_problems, d)

        # PCA reduction
        n_comp = min(N_PCA_LAYER, n_problems - 1, cloud.shape[1])
        pca = PCA(n_components=n_comp)
        cloud_pca = pca.fit_transform(cloud)

        # Persistent homology (no subsampling needed — only 30 points)
        pa = PersistenceAnalyzer(max_dim=1, backend="ripser")
        result = pa.fit_transform(cloud_pca, seed=42)

        analyzers_per_layer.append(pa)
        layer_entropies.append(result["persistence_entropy"][1])  # H1

    # Bottleneck distance between consecutive layers
    distances = []
    for ell in range(n_total_layers - 1):
        d = analyzers_per_layer[ell].distance(analyzers_per_layer[ell + 1], metric="bottleneck")
        distances.append(d)

    exp3_distances[level] = distances
    exp3_entropies[level] = layer_entropies

    print(f"  Bottleneck distances (last 5 transitions): {[f'{d:.4f}' for d in distances[-5:]]}")
    print(f"  H1 entropy (last 5 layers): {[f'{e:.4f}' for e in layer_entropies[-5:]]}")
    print()

In [ ]:
# --- Experiment 3: Plot ---

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3a. Bottleneck distance vs layer index
ax = axes[0]
colors = {"1": "tab:blue", "5": "tab:red"}
for level in LEVELS_EXP3:
    dists = exp3_distances[level]
    layer_indices = list(range(len(dists)))  # transition between layer i and i+1
    ax.plot(layer_indices, dists, "o-", color=colors[str(level)],
            linewidth=2, markersize=4, label=f"Level {level}")

# Shade the final 5 layers
n_total_layers = layer_hidden.shape[1]
shade_start = max(0, n_total_layers - 6)  # 5 transitions in last 6 layers
ax.axvspan(shade_start, n_total_layers - 2, alpha=0.1, color="gray", label="Final 5 layers")

ax.set_xlabel("Layer Transition (layer i -> i+1)")
ax.set_ylabel("Bottleneck Distance")
ax.set_title("Layer-wise Topological Transition Profile")
ax.legend()
ax.grid(True, alpha=0.3)

# 3b. H1 persistence entropy vs layer
ax = axes[1]
for level in LEVELS_EXP3:
    ents = exp3_entropies[level]
    ax.plot(range(len(ents)), ents, "o-", color=colors[str(level)],
            linewidth=2, markersize=4, label=f"Level {level}")
ax.axvspan(shade_start, n_total_layers - 1, alpha=0.1, color="gray", label="Final 5 layers")
ax.set_xlabel("Layer Index")
ax.set_ylabel("Persistence Entropy (H1)")
ax.set_title("Layer-wise H1 Complexity")
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "exp3_layerwise_transition.png"), dpi=300, bbox_inches="tight")
plt.show()

# Report terminal vs non-terminal bottleneck distances
for level in LEVELS_EXP3:
    dists = exp3_distances[level]
    if len(dists) >= 6:
        terminal = np.mean(dists[-5:])
        nonterminal = np.mean(dists[:-5])
        ratio = terminal / nonterminal if nonterminal > 0 else float("inf")
        print(f"Level {level}: terminal mean={terminal:.4f}, non-terminal mean={nonterminal:.4f}, ratio={ratio:.2f}x")

## Experiment 4: Topological Distance Between Difficulty Levels

Compute pairwise Wasserstein distances between the persistence diagrams from Experiment 1.
Test against a permutation null: shuffle difficulty labels, recompute all pairwise distances.
Ask: are the topological differences between difficulty levels significant?

In [ ]:
# --- Experiment 4: Pairwise topological distance matrix ---

from itertools import combinations

# Observed pairwise Wasserstein distances using Experiment 1 analyzers
topo_distance_matrix = np.zeros((5, 5))
for i, j in combinations(range(5), 2):
    level_i = i + 1
    level_j = j + 1
    d = exp1_analyzers[level_i].distance(exp1_analyzers[level_j], metric="wasserstein_1")
    topo_distance_matrix[i, j] = d
    topo_distance_matrix[j, i] = d

print("Observed Wasserstein distance matrix:")
header = "      " + "  ".join([f"Lv{lv}" for lv in range(1, 6)])
print(header)
for i in range(5):
    row = f"Lv{i+1}  " + "  ".join([f"{topo_distance_matrix[i, j]:.4f}" for j in range(5)])
    print(row)

# Key distances
adj_dists = [topo_distance_matrix[i, i + 1] for i in range(4)]
d_1_5 = topo_distance_matrix[0, 4]
print(f"\nMean adjacent-level distance: {np.mean(adj_dists):.4f}")
print(f"Level 1 vs Level 5 distance: {d_1_5:.4f}")

In [ ]:
# --- Experiment 4: Permutation test ---

N_PERMUTATIONS = 200

# Observed mean pairwise distance (upper triangle)
observed_mean_dist = topo_distance_matrix[np.triu_indices(5, k=1)].mean()

# Permutation null: shuffle difficulty labels, recompute PH per group, measure distances
rng = np.random.default_rng(42)
null_mean_dists = []

for perm_idx in range(N_PERMUTATIONS):
    if perm_idx % 50 == 0:
        print(f"Permutation {perm_idx}/{N_PERMUTATIONS}...")

    # Shuffle level labels
    shuffled_levels = rng.permutation(levels)

    # PCA + PH per shuffled group
    perm_analyzers = {}
    for level in range(1, 6):
        mask = shuffled_levels == level
        cloud = last_hidden[mask]
        n_k = cloud.shape[0]
        if n_k < 3:
            perm_analyzers[level] = None
            continue

        n_comp = min(N_PCA_COMPONENTS, n_k - 1, cloud.shape[1])
        pca = PCA(n_components=n_comp)
        cloud_pca = pca.fit_transform(cloud)

        pa = PersistenceAnalyzer(max_dim=2, backend="ripser")
        pa.fit_transform(cloud_pca, subsample=min(n_k, 200), seed=42)
        perm_analyzers[level] = pa

    # Pairwise distances
    perm_dists = []
    for i, j in combinations(range(5), 2):
        pa_i = perm_analyzers[i + 1]
        pa_j = perm_analyzers[j + 1]
        if pa_i is not None and pa_j is not None:
            d = pa_i.distance(pa_j, metric="wasserstein_1")
            perm_dists.append(d)
    if perm_dists:
        null_mean_dists.append(np.mean(perm_dists))

null_mean_dists = np.array(null_mean_dists)

# p-value: fraction of permutations with mean distance >= observed
p_value = (np.sum(null_mean_dists >= observed_mean_dist) + 1) / (len(null_mean_dists) + 1)
z_score = (observed_mean_dist - null_mean_dists.mean()) / (null_mean_dists.std() + 1e-15)

print(f"\nObserved mean pairwise Wasserstein distance: {observed_mean_dist:.4f}")
print(f"Null distribution: mean={null_mean_dists.mean():.4f}, std={null_mean_dists.std():.4f}")
print(f"z-score: {z_score:.2f}")
print(f"Permutation p-value: {p_value:.4f}")
print(f"Significant at alpha=0.05: {p_value < 0.05}")

In [ ]:
# --- Experiment 4: Plots ---

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 4a. Distance matrix heatmap
ax = axes[0]
im = ax.imshow(topo_distance_matrix, cmap="YlOrRd", origin="lower")
ax.set_xticks(range(5))
ax.set_xticklabels([f"Level {i}" for i in range(1, 6)])
ax.set_yticks(range(5))
ax.set_yticklabels([f"Level {i}" for i in range(1, 6)])
ax.set_title("Pairwise Wasserstein Distance (H0+H1)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
# Annotate cells
for i in range(5):
    for j in range(5):
        val = topo_distance_matrix[i, j]
        color = "white" if val > topo_distance_matrix.max() * 0.6 else "black"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=8, color=color)

# 4b. Permutation null distribution
ax = axes[1]
ax.hist(null_mean_dists, bins=30, alpha=0.7, color="steelblue", edgecolor="white")
ax.axvline(observed_mean_dist, color="red", linewidth=2, label=f"Observed = {observed_mean_dist:.4f}")
p95 = np.percentile(null_mean_dists, 95)
ax.axvline(p95, color="orange", linewidth=1.5, linestyle="--", label=f"95th pctile = {p95:.4f}")
ax.set_xlabel("Mean Pairwise Wasserstein Distance")
ax.set_ylabel("Count")
ax.set_title(f"Permutation Null (n={N_PERMUTATIONS}), p={p_value:.4f}")
ax.legend()

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "exp4_distance_matrix.png"), dpi=300, bbox_inches="tight")
plt.show()

## Final Summary

Consolidated table of all metrics across difficulty levels, plus cross-metric correlations.

In [ ]:
# --- Final Summary Table ---

# Collect all per-level scalar metrics
summary = {}
for level in range(1, 6):
    # Sparsity baselines
    sm = sparsity_metrics[level]

    # Exp 1: persistence entropy H1, PCA variance @50
    pe_h1 = exp1_results[level]["persistence_entropy"][1]
    cv = exp1_pca_info[level]
    pca_var_50 = cv[min(49, len(cv) - 1)]

    # Exp 2: condition number, degeneracy rate (from trajectory analysis)
    e2 = exp2_results[level]
    if e2:
        mean_cond = np.mean([r["condition_number"] for r in e2])
        degen_rate = np.mean([r["degenerate"] for r in e2])
    else:
        mean_cond = float("nan")
        degen_rate = float("nan")

    summary[level] = {
        "l1_norm": sm["l1_norm"],
        "top10_energy": sm["top10_energy"],
        "eff_rank": sm["eff_rank"],
        "pers_entropy_h1": pe_h1,
        "pca_var_50": pca_var_50,
        "cond_number": mean_cond,
        "degen_rate": degen_rate,
    }

# Print table
cols = ["L1 Norm", "Top10% E", "Eff Rank", "PE(H1)", "PCA@50", "Cond #", "Degen %"]
keys = ["l1_norm", "top10_energy", "eff_rank", "pers_entropy_h1", "pca_var_50", "cond_number", "degen_rate"]

header = "Level | " + " | ".join(f"{c:>9s}" for c in cols)
sep = "------|" + "|".join(["-" * 11] * len(cols))
print(header)
print(sep)
for level in range(1, 6):
    vals = []
    for k in keys:
        v = summary[level][k]
        if k == "degen_rate":
            vals.append(f"{v:>8.0%} " if not np.isnan(v) else "      n/a ")
        elif k == "cond_number":
            vals.append(f"{v:>9.1f}" if not np.isnan(v) else "      n/a")
        else:
            vals.append(f"{v:>9.4f}")
    print(f"  {level}   | " + " | ".join(vals))

# Correlation matrix
print("\n\nPearson Correlation Matrix (n=5, one value per difficulty level):")
metric_names = ["L1 Norm", "Top10% Energy", "Eff Rank", "PE(H1)", "PCA Var@50", "Cond Number"]
metric_keys = ["l1_norm", "top10_energy", "eff_rank", "pers_entropy_h1", "pca_var_50", "cond_number"]
metric_arrays = {}
for k in metric_keys:
    arr = np.array([summary[lv][k] for lv in range(1, 6)])
    metric_arrays[k] = arr

# Print correlation pairs
print(f"\n{'':>16s}", end="")
for name in metric_names:
    print(f" {name:>13s}", end="")
print()

for i, (name_i, key_i) in enumerate(zip(metric_names, metric_keys)):
    print(f"{name_i:>16s}", end="")
    for j, (name_j, key_j) in enumerate(zip(metric_names, metric_keys)):
        arr_i = metric_arrays[key_i]
        arr_j = metric_arrays[key_j]
        if np.any(np.isnan(arr_i)) or np.any(np.isnan(arr_j)):
            print(f" {'n/a':>13s}", end="")
        else:
            r, _ = stats.pearsonr(arr_i, arr_j)
            print(f" {r:>13.3f}", end="")
    print()

print("\n\nAll figures saved to:", FIGURES_DIR)